# Explanation: why bother combinining values from multiple regions in the same cell?

## Code setup

In [1]:
# Setting up paths to files:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd  # to load props dataframe

# Add path to package:
import sys
sys.path.insert(0,'../')
# import isopods.grid

In [2]:
@dataclass(frozen=True)
class Paths:
    '''Singleton object for storing paths to data and database.'''

    dir_data: str = '../data'
    grid_filename: str = 'lsoa2011_devon_1k_grid_df.csv'
    lookup_filename: str = 'lookup_lsoa2011_district_devon.csv'

paths = Paths()

## Load in grid data

In [3]:
# Proportions dataframe:
p = os.path.join(paths.dir_data, paths.grid_filename)
# Use converters to turn proportion data from strings to lists.
converters = {
    'label_props': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
    'area_props': lambda x: [float(f) for f in x[1:-1].split()]
}
df_props = pd.read_csv(p, index_col=0, converters=converters)

# Confirm that contents are as expected:
for col in ['label_props', 'area_props']:
    print(col)
    print('Data type:', type(df_props.iloc[0][col]))
    print('List value type:', type(df_props.iloc[0][col][0]))

label_props
Data type: <class 'list'>
List value type: <class 'str'>
area_props
Data type: <class 'list'>
List value type: <class 'float'>


In [4]:
df_props.head(3)

,cell_row,cell_col,label_props,area_props,prop_cell_covered,label_majority,overlapping_polys
cell_label,,,,,,,
109,109,0,[E01020288],[1.0],0.000,E01020288,0
110,110,0,[E01020288],[1.0],0.124,E01020288,0
111,111,0,[E01020288],[1.0],0.089,E01020288,0


Pick out the labels of cells that contain many regions:

In [5]:
inds_many_labels = df_props[df_props['label_props'].str.len() > 3].index

## Check whether all regions are represented in majority grid

Do all regions appear in the majority grid? Are there regions that cover only a bit less area than the largest region in the cell?

Load in this region lookup which contains all of the region names:

In [6]:
# Load in LSOA to district name lookup:
p = os.path.join(paths.dir_data, paths.lookup_filename)
dict_lookup = pd.read_csv(p, index_col='LSOA11CD').squeeze().to_dict()

Compare the full list with the list in our grid. How many regions are not represented in the grid?

In [7]:
lsoa_maj = df_props['label_majority'].unique()
n_lsoa_total = len(list(dict_lookup.keys()))
missing_lsoa = set(list(dict_lookup.keys())) - set(lsoa_maj)
print(f'{len(missing_lsoa)} of {n_lsoa_total} regions are missing.')

264 of 707 regions are missing.


These missing regions are in the full proportions data, just never as the majority region. Pick out information about where the regions rank in each cell.

First expand the lists in the label proportions column. Change from a dataframe with one row per cell to another dataframe with one row per cell-region combo. For example, a cell with six regions in it will take six rows in this new dataframe.

In [8]:
df = pd.concat(
        [df_props.explode('label_props').drop('area_props', axis='columns'),
         df_props[['area_props']].explode('area_props')],
        axis='columns'
    )

In [9]:
print(len(df_props), len(df))

7101 12258


View the rows for one cell with many regions:

In [10]:
cols_to_show = ['label_props', 'area_props']
for d in [df_props, df]:
    display(d.loc[inds_many_labels[0], cols_to_show])

label_props    [E01015142, E01015083, E01015150, E01015146, E...
area_props                       [0.779, 0.178, 0.042, 0.0, 0.0]
Name: 3682, dtype: object

,label_props,area_props
cell_label,,
3682,E01015142,0.779
3682,E01015083,0.178
3682,E01015150,0.042
3682,E01015146,0.0
3682,E01015145,0.0


Which missing regions appear in multiple cells?

In [11]:
mask = df['label_props'].isin(missing_lsoa)
display(df.loc[mask, 'label_props'].value_counts().head())

label_props
E01020089    6
E01015191    6
E01020065    6
E01019930    6
E01015172    5
Name: count, dtype: int64

Some of these regions appear in six cells but are never the largest region in the cell.

### Proportion

See the largest proportion that the missing regions cover in any cell:

In [12]:
df_dd = (df[df['label_props'].isin(missing_lsoa)]
         .sort_values('area_props', ascending=False)
         .drop_duplicates('label_props'))

display(df_dd[['label_props', 'area_props']].head(10))

,label_props,area_props
cell_label,,
8759,E01015191,0.491
9112,E01015232,0.479
10693,E01020055,0.456
13275,E01019899,0.437
3987,E01020282,0.416
4935,E01020085,0.415
9469,E01015267,0.392
10574,E01020058,0.389
11853,E01019956,0.388


Nine of the regions that are missing from the majority grid cover over 40% of the area of their cell.

Are the missing regions always the second-largest in the cell?

### Ranking regions

Define this function to rank the region standings in each cell.

In [13]:
def find_order_of_regions_in_cell(df, col='cell_label'):
    """
    Example input df: df_props.explode('label_props')
    where there is also a column called col.
    Input should be exploded by region label, i.e.
    each cell label may appear multiple times if that cell
    contains multiple regions.
    Each row is a different combination of cell and region labels.
    """
    # Set up columns for each cell label.
    # Values are 1 where the cell label is used by this row.
    df_cell_counts_bin = pd.get_dummies(df[col])
    # Find how many times this cell has been used so far:
    df_cell_counts = (
        df_cell_counts_bin.cumsum(axis='rows') *
        df_cell_counts_bin
    )
    # In each row, only one column is non-zero. Use sum to gather
    # non-zero values:
    df_cell_counts = df_cell_counts.sum(axis='columns')
    df_cell_counts.name = f'{col}_count'

    # Place this new column into the original dataframe.
    df_cell_counts = pd.concat((df, df_cell_counts), axis='columns')
    return df_cell_counts

In [14]:
df = find_order_of_regions_in_cell(df.reset_index(), col='cell_label')
df = df.set_index('cell_label')

View the results for one cell with many regions:

In [15]:
cols_to_show = ['label_props', 'area_props', 'cell_label_count']
display(df.loc[inds_many_labels[0], cols_to_show])

,label_props,area_props,cell_label_count
cell_label,,,
3682,E01015142,0.779,1
3682,E01015083,0.178,2
3682,E01015150,0.042,3
3682,E01015146,0.0,4
3682,E01015145,0.0,5


How many regions appear at each rank of the area proportions?

Only count each region once. If its best standing is second in the list, and it also appears in positions four and five in other cells, then count the region only as second-largest.

In [16]:
df_dd = df.sort_values('cell_label_count').drop_duplicates('label_props')

cell_label_count_min = df_dd['cell_label_count'].min()
cell_label_count_max = df_dd['cell_label_count'].max()

s_count_count = {}
for n in range(cell_label_count_min, cell_label_count_max+1, 1):
    m = df_dd['cell_label_count'] == n
    s_count_count[n] = len(df_dd[m])
s_count_count = pd.Series(s_count_count, name='n_regions_in_this_rank')

display(s_count_count)

1    443
2    153
3     66
4     30
5     10
6      3
7      1
8      1
Name: n_regions_in_this_rank, dtype: int64

Most of the missing regions are the second-biggest region in a cell.

One region is only ever the eighth-biggest region! Check these smallest regions:

In [17]:
display(df_dd[['label_props', 'area_props', 'cell_label_count']].tail())

,label_props,area_props,cell_label_count
cell_label,,,
4268,E01015054,0.069,6
9379,E01020031,0.098,6
9380,E01020029,0.072,6
9617,E01019997,0.067,7
4268,E01015053,0.049,8


Check how many regions are in the cell with the smallest missing region:

In [18]:
display(df.loc[4268, ['label_props', 'area_props', 'cell_label_count']])

,label_props,area_props,cell_label_count
cell_label,,,
4268,E01032658,0.199,1
4268,E01015181,0.197,2
4268,E01032657,0.177,3
4268,E01015182,0.146,4
4268,E01015179,0.088,5
4268,E01015054,0.069,6
4268,E01015156,0.063,7
4268,E01015053,0.049,8
4268,E01015178,0.009,9


Loads of regions! This cell is most likely in a cluster of small regions, such as in a city.